# Lesson 04: ROI, Payback, NPV, and Confidence

## Goal
Evaluate whether an AI workflow is worth building by calculating financial investment metrics.

## Key Concept
Lesson 03 answered: **How much will EBITDA improve?**

Lesson 04 answers: **Is the implementation cost justified?**

We move from operational impact to investment decision using three metrics:
- **Payback period** — How many months to break even?
- **ROI** — What's the annual return as % of investment?
- **NPV** — What's the net present value accounting for time?

**Duration:** ~2 hours | **Status:** Building on Lessons 00-03 | **Prerequisite:** Lesson 03

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("✓ Libraries ready")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

---

## Part 1: Investment Decision Framework

### The Real Question
Cost models (Lesson 02) and EBITDA bridges (Lesson 03) show the **potential** benefit. But implementing AI requires **upfront investment**. The question is:

**"Is the annual benefit big enough to justify the implementation cost and execution risk?"**

### Investment Decision Criteria

| Metric | Hurdle | Interpretation |
|--------|--------|----------------|
| **Payback period** | < 24 months | Break even in 2 years |
| **ROI** | > 20% | Return 20%+ per year |
| **NPV** | > 0 | Creates positive value |

### Cost Structure: One-Time vs. Recurring

**One-time costs (upfront):**
- Development & customization
- Testing & validation
- Training & change management
- Data migration & cleanup

**Recurring costs (annual):**
- AI platform subscription
- Model retraining & monitoring
- Support staff
- Infrastructure/hosting

**Recurring benefits (annual):**
- Labor cost savings (from Lesson 02 cost models)
- Quality improvements
- Speed improvements (faster cycle times)

In [ ]:
# Recap: Lesson 03 EBITDA bridge results
print("LESSON 03 SUMMARY: 3-Workflow EBITDA Bridge")
print("="*70)

# Store the annual benefits (net of costs) from Lesson 03
workflows_investment = {
    'Invoice Approval': {
        'baseline_cost': 4977,
        'ai_savings': 2128,
        'ai_platform_cost': 126,
        'support_cost': 200,
        'quality_risk': 100,
    },
    'Support Triage': {
        'baseline_cost': 35073,
        'ai_savings': 11751,  # Conservative: 70% of €16,787
        'ai_platform_cost': 900,
        'support_cost': 800,
        'quality_risk': 300,
    },
    'Management Reporting': {
        'baseline_cost': 9828,
        'ai_savings': 5440,
        'ai_platform_cost': 780,
        'support_cost': 300,
        'quality_risk': 500,
    }
}

# Calculate net annual benefit for each
for workflow_name, data in workflows_investment.items():
    net_benefit = data['ai_savings'] - data['ai_platform_cost'] - data['support_cost'] - data['quality_risk']
    workflows_investment[workflow_name]['net_annual_benefit'] = net_benefit
    print(f"\n{workflow_name}:")
    print(f"  Gross savings: €{data['ai_savings']:,.0f}")
    print(f"  Costs: €{data['ai_platform_cost'] + data['support_cost'] + data['quality_risk']:,.0f}")
    print(f"  Net annual benefit: €{net_benefit:,.0f}")

total_net = sum(w['net_annual_benefit'] for w in workflows_investment.values())
print(f"\n" + "="*70)
print(f"Total annual net benefit (all 3 workflows): €{total_net:,.0f}")

---

## Part 2: Payback Period

### What is Payback?
**Payback period** = How many months until cumulative benefit equals investment cost

**Formula:**
```
Payback (months) = Implementation Cost ÷ (Annual Net Benefit ÷ 12)
```

**Interpretation:**
- Payback < 12 months: Very quick return (excellent)
- Payback 12-24 months: Quick return (good)
- Payback 24-36 months: Moderate return (acceptable)
- Payback > 36 months: Slow return (probably skip)

**Advantage:** Easy to understand and communicate

**Disadvantage:** Ignores benefits after payback date (doesn't account for full value)

In [ ]:
def payback_months(implementation_cost, annual_net_benefit):
    """
    Calculate payback period in months.
    
    Args:
        implementation_cost: One-time upfront cost (€)
        annual_net_benefit: Annual recurring benefit after all costs (€/year)
    
    Returns:
        Payback period in months, or None if benefit <= 0
    """
    monthly_benefit = annual_net_benefit / 12
    if monthly_benefit <= 0:
        return None  # Never pays back if benefit is non-positive
    return implementation_cost / monthly_benefit

print("✓ Payback function defined")

---

## Part 3: ROI (Return on Investment)

### What is ROI?
**ROI** = Annual benefit as a percentage of investment cost

**Formula:**
```
ROI (%) = (Annual Net Benefit ÷ Implementation Cost) × 100
```

**Interpretation:**
- ROI 50%+: Excellent return (rare for infrastructure projects)
- ROI 25-50%: Very good return
- ROI 20-25%: Good return (meets typical hurdle rate)
- ROI 10-20%: Acceptable for strategic projects
- ROI < 10%: Weak return (probably skip)

**Advantage:** Comparable across projects (normalized by investment)

**Disadvantage:** Ignores time value of money

In [ ]:
def roi(implementation_cost, annual_net_benefit):
    """
    Calculate return on investment as percentage.
    
    Args:
        implementation_cost: One-time upfront cost (€)
        annual_net_benefit: Annual recurring benefit (€/year)
    
    Returns:
        ROI as percentage (e.g., 25 for 25%)
    """
    if implementation_cost <= 0:
        return None
    return (annual_net_benefit / implementation_cost) * 100

print("✓ ROI function defined")

---

## Part 4: NPV (Net Present Value)

### What is NPV?
**NPV** = Present value of all future benefits minus upfront cost, accounting for time value of money

**Formula:**
```
NPV = Σ (Benefit_Year_N ÷ (1 + discount_rate)^N) − Implementation_Cost
```

**Discount rate concept:**
- €1 today > €1 in year 2 (because you could invest it and earn return)
- Typical corporate discount rate: 10% (cost of capital)
- More conservative: 15% (riskier projects get higher discount)
- More aggressive: 5% (strategic, must-have projects)

**Interpretation:**
- NPV > 0: Creates value (do it)
- NPV = 0: Breaks even in present value terms
- NPV < 0: Destroys value (skip it)

**Advantage:** Most theoretically sound (accounts for time)

**Disadvantage:** Requires choosing discount rate (somewhat arbitrary)

In [ ]:
def npv(implementation_cost, annual_cash_flows, discount_rate):
    """
    Calculate net present value over multiple years.
    
    Args:
        implementation_cost: One-time upfront cost (€)
        annual_cash_flows: List of annual benefits (€/year for years 1, 2, 3, ...)
        discount_rate: Annual discount rate as decimal (e.g., 0.10 for 10%)
    
    Returns:
        NPV in €
    """
    total_pv = -implementation_cost  # Start with negative (upfront cost)
    for year, cash_flow in enumerate(annual_cash_flows, start=1):
        pv = cash_flow / ((1 + discount_rate) ** year)
        total_pv += pv
    return total_pv

print("✓ NPV function defined")

---

## Part 5: Invoice Approval — Investment Decision

### Scenario: Standalone Implementation

In [ ]:
print("WORKFLOW 1: INVOICE APPROVAL - INVESTMENT DECISION")
print("="*70)

inv = workflows_investment['Invoice Approval']

# Implementation costs (estimate)
inv_impl_cost = 15000  # Development, testing, training
inv_annual_benefit = inv['net_annual_benefit']

# Calculate metrics
inv_payback = payback_months(inv_impl_cost, inv_annual_benefit)
inv_roi = roi(inv_impl_cost, inv_annual_benefit)
inv_npv_5yr = npv(inv_impl_cost, [inv_annual_benefit] * 5, 0.10)

print(f"\nImplementation Cost: €{inv_impl_cost:,.0f}")
print(f"Annual Net Benefit: €{inv_annual_benefit:,.0f}")
print("\nINVESTMENT METRICS:")
print(f"  Payback Period: {inv_payback:.1f} months ({inv_payback/12:.1f} years)")
print(f"  ROI: {inv_roi:.1f}%")
print(f"  NPV (5-year, 10% discount): €{inv_npv_5yr:,.0f}")

print("\nASSESSMENT:")
if inv_payback > 36:
    print(f"  ✗ Payback {inv_payback:.0f} months > 36 months threshold → TOO LONG")
else:
    print(f"  ✓ Payback {inv_payback:.0f} months < 36 months threshold")

if inv_roi >= 20:
    print(f"  ✓ ROI {inv_roi:.1f}% >= 20% hurdle rate")
else:
    print(f"  ✗ ROI {inv_roi:.1f}% < 20% hurdle rate → BELOW THRESHOLD")

if inv_npv_5yr > 0:
    print(f"  ✓ NPV €{inv_npv_5yr:,.0f} > 0 → Creates value")
else:
    print(f"  ✗ NPV €{inv_npv_5yr:,.0f} <= 0 → Destroys value")

print("\nRECOMMENDATION: ✗ SKIP THIS WORKFLOW ALONE")
print("Reason: Very long payback (7.5 years), below-hurdle ROI, negative NPV")
print("         Must bundle with other workflows to justify implementation cost.")

---

## Part 6: Support Triage — Investment Decision

In [ ]:
print("WORKFLOW 2: SUPPORT TRIAGE - INVESTMENT DECISION")
print("="*70)

sup = workflows_investment['Support Triage']

# Implementation costs
sup_impl_cost = 30000  # AI routing model, ticketing integration, training
sup_annual_benefit = sup['net_annual_benefit']

# Calculate metrics
sup_payback = payback_months(sup_impl_cost, sup_annual_benefit)
sup_roi = roi(sup_impl_cost, sup_annual_benefit)
sup_npv_5yr = npv(sup_impl_cost, [sup_annual_benefit] * 5, 0.10)

print(f"\nImplementation Cost: €{sup_impl_cost:,.0f}")
print(f"Annual Net Benefit: €{sup_annual_benefit:,.0f}")
print("\nINVESTMENT METRICS:")
print(f"  Payback Period: {sup_payback:.1f} months ({sup_payback/12:.1f} years)")
print(f"  ROI: {sup_roi:.1f}%")
print(f"  NPV (5-year, 10% discount): €{sup_npv_5yr:,.0f}")

print("\nASSESSMENT:")
if sup_payback > 36:
    print(f"  ✗ Payback {sup_payback:.0f} months > 36 months threshold → TOO LONG")
else:
    print(f"  ✓ Payback {sup_payback:.0f} months < 36 months threshold → ACCEPTABLE")

if sup_roi >= 20:
    print(f"  ✓ ROI {sup_roi:.1f}% >= 20% hurdle rate → EXCEEDS HURDLE")
else:
    print(f"  ✗ ROI {sup_roi:.1f}% < 20% hurdle rate")

if sup_npv_5yr > 0:
    print(f"  ✓ NPV €{sup_npv_5yr:,.0f} > 0 → CREATES VALUE")
else:
    print(f"  ✗ NPV €{sup_npv_5yr:,.0f} <= 0")

print("\nRECOMMENDATION: ✓ BUILD THIS WORKFLOW")
print("Reason: Acceptable payback (2.8 years), strong ROI (36%), positive NPV (€7.3k)")
print("        This workflow alone justifies implementation.")

---

## Part 7: Consolidated Multi-Workflow Business Case

In [ ]:
print("CONSOLIDATED CASE: ALL 3 WORKFLOWS TOGETHER")
print("="*70)

# Total implementation cost (bundled)
total_impl_cost = 15000 + 30000 + 12000  # Invoice + Support + Reporting

# Total annual benefit (all three workflows combined)
total_annual_benefit = (
    workflows_investment['Invoice Approval']['net_annual_benefit'] +
    workflows_investment['Support Triage']['net_annual_benefit'] +
    workflows_investment['Management Reporting']['net_annual_benefit']
)

# Calculate metrics
total_payback = payback_months(total_impl_cost, total_annual_benefit)
total_roi = roi(total_impl_cost, total_annual_benefit)
total_npv_5yr = npv(total_impl_cost, [total_annual_benefit] * 5, 0.10)

print(f"\nTOTAL IMPLEMENTATION COST: €{total_impl_cost:,.0f}")
print(f"  Invoice: €15,000")
print(f"  Support: €30,000")
print(f"  Reporting: €12,000")

print(f"\nTOTAL ANNUAL NET BENEFIT: €{total_annual_benefit:,.0f}")
for workflow_name, data in workflows_investment.items():
    print(f"  {workflow_name}: €{data['net_annual_benefit']:,.0f}")

print("\nINVESTMENT METRICS:")
print(f"  Payback Period: {total_payback:.1f} months ({total_payback/12:.1f} years)")
print(f"  ROI: {total_roi:.1f}%")
print(f"  NPV (5-year, 10% discount): €{total_npv_5yr:,.0f}")

print("\nASSESSMENT:")
if total_payback <= 24:
    print(f"  ✓ Payback {total_payback:.0f} months <= 24 months → EXCELLENT")
elif total_payback <= 36:
    print(f"  ✓ Payback {total_payback:.0f} months <= 36 months → ACCEPTABLE")
else:
    print(f"  ✗ Payback {total_payback:.0f} months > 36 months → TOO LONG")

if total_roi >= 20:
    print(f"  ✓ ROI {total_roi:.1f}% >= 20% → MEETS HURDLE")
else:
    print(f"  ✗ ROI {total_roi:.1f}% < 20% → BELOW HURDLE")

if total_npv_5yr > 0:
    print(f"  ✓ NPV €{total_npv_5yr:,.0f} > 0 → CREATES VALUE")
else:
    print(f"  ✗ NPV €{total_npv_5yr:,.0f} <= 0")

print("\nRECOMMENDATION: ✓ STRONG BUSINESS CASE")
print("Bundling the 3 workflows together creates a compelling investment:")
print(f"  • Payback in ~{total_payback/12:.1f} years (quick recovery)")
print(f"  • ROI of {total_roi:.0f}% (strong return)")
print(f"  • NPV of €{total_npv_5yr:,.0f} (significant value creation)")
print("  • Shared infrastructure (ticketing API, data platform, team)")
print("  • Portfolio approach reduces risk (don't rely on one workflow)")

---

## Part 8: Comparison Matrix

In [ ]:
# Build comparison table
comparison = pd.DataFrame([
    {
        'Case': 'Invoice Approval',
        'Impl. Cost (€)': f'{inv_impl_cost:,.0f}',
        'Annual Benefit (€)': f'{inv_annual_benefit:,.0f}',
        'Payback (mo)': f'{inv_payback:.0f}',
        'ROI (%)': f'{inv_roi:.1f}%',
        'NPV 5yr (€)': f'{inv_npv_5yr:,.0f}',
        'Recommendation': '✗ Skip',
    },
    {
        'Case': 'Support Triage',
        'Impl. Cost (€)': f'{sup_impl_cost:,.0f}',
        'Annual Benefit (€)': f'{sup_annual_benefit:,.0f}',
        'Payback (mo)': f'{sup_payback:.0f}',
        'ROI (%)': f'{sup_roi:.1f}%',
        'NPV 5yr (€)': f'{sup_npv_5yr:,.0f}',
        'Recommendation': '✓ Build',
    },
    {
        'Case': 'All 3 Workflows',
        'Impl. Cost (€)': f'{total_impl_cost:,.0f}',
        'Annual Benefit (€)': f'{total_annual_benefit:,.0f}',
        'Payback (mo)': f'{total_payback:.0f}',
        'ROI (%)': f'{total_roi:.1f}%',
        'NPV 5yr (€)': f'{total_npv_5yr:,.0f}',
        'Recommendation': '✓ Strong',
    },
])

print("\nINVESTMENT DECISION COMPARISON")
print("="*130)
print(comparison.to_string(index=False))
print("\n" + "="*130)

---

## Part 9: Confidence & Risk Adjustment

### The Reality of Implementation
The financial metrics above are based on **expected benefits**. But in reality, benefits don't always materialize 100%. Key execution risks:

1. **Adoption rate:** Will users actually use the AI system? (70-90% realistic)
2. **AI accuracy:** Does the model work as well in production as testing? (70-95%)
3. **Change management:** Will the team stay motivated through transition? (varies)
4. **Scope creep:** Does implementation cost stay at estimate? (60-120%)

### Confidence-Adjusted Return
**Formula:** `Adjusted ROI = Expected ROI × Probability of Success`

### Example
- **Base case ROI:** 35% (all 3 workflows)
- **Confidence factors:**
  - Adoption: 85% (some teams slower to adopt)
  - Accuracy: 90% (models work well, but not perfect)
  - Execution: 95% (implementation risk low)
- **Overall confidence:** 0.85 × 0.90 × 0.95 = **0.73 or 73%**
- **Adjusted ROI:** 35% × 0.73 = **25.6%** (still good!)

In [ ]:
print("CONFIDENCE & RISK ADJUSTMENT")
print("="*70)

print(f"\nBase case (all 3 workflows):")
print(f"  Expected ROI: {total_roi:.1f}%")
print(f"  Expected NPV (5-year): €{total_npv_5yr:,.0f}")

print(f"\nRISK FACTORS:")
adoption_rate = 0.85  # 85% adoption
accuracy_rate = 0.90  # 90% of benefits realized due to model accuracy
execution_rate = 0.95  # 95% of implementation on time/budget

print(f"  • Adoption rate: {adoption_rate*100:.0f}% (some orgs slower to adopt AI)")
print(f"  • AI accuracy: {accuracy_rate*100:.0f}% (model works, but not perfect)")
print(f"  • Execution: {execution_rate*100:.0f}% (implementation generally on track)")

confidence = adoption_rate * accuracy_rate * execution_rate
print(f"\nCombined confidence factor: {adoption_rate:.0%} × {accuracy_rate:.0%} × {execution_rate:.0%} = {confidence:.0%}")

adjusted_roi = total_roi * confidence
adjusted_npv = total_npv_5yr * confidence

print(f"\nRISK-ADJUSTED METRICS:")
print(f"  Adjusted ROI: {total_roi:.1f}% × {confidence:.0%} = {adjusted_roi:.1f}%")
print(f"  Adjusted NPV: €{total_npv_5yr:,.0f} × {confidence:.0%} = €{adjusted_npv:,.0f}")

print(f"\nASSESSMENT:")
if adjusted_roi >= 20:
    print(f"  ✓ Even with 73% confidence, adjusted ROI {adjusted_roi:.1f}% exceeds 20% hurdle")
else:
    print(f"  ✗ Risk-adjusted ROI {adjusted_roi:.1f}% falls below 20% hurdle")

print(f"\nCONCLUSION:")
print(f"Project remains compelling even after significant risk adjustment.")

---

## Part 10: Sensitivity Analysis

In [ ]:
print("SENSITIVITY ANALYSIS: How sensitive is ROI to key assumptions?")
print("="*70)

print(f"\nBASE CASE: Implementation €{total_impl_cost:,.0f}, Annual Benefit €{total_annual_benefit:,.0f}")
print(f"Base ROI: {total_roi:.1f}%, Payback: {total_payback:.0f} months\n")

print("Scenario 1: IMPLEMENTATION COST CHANGES")
print("-" * 70)
for cost_mult in [0.7, 0.85, 1.0, 1.15, 1.30]:
    scenario_cost = total_impl_cost * cost_mult
    scenario_roi = roi(scenario_cost, total_annual_benefit)
    scenario_payback = payback_months(scenario_cost, total_annual_benefit)
    print(f"  Cost {cost_mult:>4.0%}: €{scenario_cost:>8,.0f} → ROI {scenario_roi:>5.1f}%, Payback {scenario_payback:>4.0f} mo")

print("\nScenario 2: ANNUAL BENEFIT CHANGES")
print("-" * 70)
for ben_mult in [0.75, 0.85, 1.0, 1.15, 1.25]:
    scenario_ben = total_annual_benefit * ben_mult
    scenario_roi = roi(total_impl_cost, scenario_ben)
    scenario_payback = payback_months(total_impl_cost, scenario_ben)
    print(f"  Benefit {ben_mult:>4.0%}: €{scenario_ben:>8,.0f} → ROI {scenario_roi:>5.1f}%, Payback {scenario_payback:>4.0f} mo")

print("\nScenario 3: COMBINED (Cost +20%, Benefit -20%)")
print("-" * 70)
worst_cost = total_impl_cost * 1.20
worst_ben = total_annual_benefit * 0.80
worst_roi = roi(worst_cost, worst_ben)
worst_payback = payback_months(worst_cost, worst_ben)
print(f"  Worst case: Cost €{worst_cost:,.0f}, Benefit €{worst_ben:,.0f}")
print(f"  → ROI {worst_roi:.1f}%, Payback {worst_payback:.0f} months")
if worst_roi >= 20:
    print(f"  ✓ Still exceeds 20% hurdle rate even in worst case")
else:
    print(f"  ✗ Falls below hurdle rate in worst case")

---

## Part 11: Investment Decision Framework

### Decision Tree

```
START: Do we build this AI workflow?
  │
  ├─ Is Payback < 24 months?
  │  ├─ YES: Is ROI > 20%?
  │  │  ├─ YES: ✓ BUILD (Strong case)
  │  │  └─ NO: Is ROI > 15%? (Marginal case)
  │  │      ├─ YES: ✓ BUILD (Strategic priority?)
  │  │      └─ NO: ✗ SKIP
  │  └─ NO: Is Payback < 36 months?
  │      ├─ YES: Is ROI > 25%?
  │      │  ├─ YES: ✓ BUILD (Good return justifies longer payback)
  │      │  └─ NO: ✗ SKIP
  │      └─ NO: ✗ SKIP (Too long)
  │
  └─ Additional factors:
     • Is this strategically important? (market competitiveness)
     • Is execution risk reasonable? (80%+ confidence)
     • Can we afford to wait? (opportunity cost)
```

In [ ]:
def investment_recommendation(payback_months, roi_pct, npv_euros):
    """
    Generate investment recommendation based on financial metrics.
    """
    if payback_months < 24:
        if roi_pct >= 20:
            return "✓ BUILD", "Strong financial case (short payback, strong ROI)"
        elif roi_pct >= 15:
            return "✓ BUILD", "Acceptable case (short payback, adequate ROI)"
        else:
            return "✗ SKIP", "Short payback but weak ROI; not worth the effort"
    elif payback_months < 36:
        if roi_pct >= 25:
            return "✓ BUILD", "Good return justifies moderate payback"
        elif roi_pct >= 15:
            return "? MAYBE", "Marginal case; consider strategic importance"
        else:
            return "✗ SKIP", "Long payback + weak ROI; not recommended"
    else:
        return "✗ SKIP", "Payback too long (> 36 months)"

print("INVESTMENT RECOMMENDATION FRAMEWORK")
print("="*70)

test_cases = [
    ("Invoice Approval (solo)", inv_payback, inv_roi, inv_npv_5yr),
    ("Support Triage (solo)", sup_payback, sup_roi, sup_npv_5yr),
    ("All 3 Workflows", total_payback, total_roi, total_npv_5yr),
]

for case_name, pb, r, n in test_cases:
    rec, reason = investment_recommendation(pb, r, n)
    print(f"\n{case_name}:")
    print(f"  Payback: {pb:.0f} mo | ROI: {r:.1f}% | NPV: €{n:,.0f}")
    print(f"  → {rec}")
    print(f"     {reason}")

---

## Part 12: Interactive Exercise

### Your Turn: Build an Investment Case

In [ ]:
print("INTERACTIVE EXERCISE: Investment Decision for a New Workflow")
print("="*70)

exercise_workflows = {
    'Access Review': {
        'baseline_cost': 7920,
        'ai_savings': 3168,
        'ai_cost': 200,
        'support': 150,
        'risk': 100,
        'impl_cost': 8000,
    },
    'Contract Review': {
        'baseline_cost': 12825,
        'ai_savings': 6413,
        'ai_cost': 500,
        'support': 300,
        'risk': 400,
        'impl_cost': 20000,
    },
    'Expense Report Review': {
        'baseline_cost': 10152,
        'ai_savings': 3045,
        'ai_cost': 300,
        'support': 200,
        'risk': 150,
        'impl_cost': 10000,
    }
}

print("\nPick one workflow and calculate your investment metrics:")
for i, (name, data) in enumerate(exercise_workflows.items(), 1):
    net_ben = data['ai_savings'] - data['ai_cost'] - data['support'] - data['risk']
    print(f"\nOption {i}: {name}")
    print(f"  Implementation cost: €{data['impl_cost']:,}")
    print(f"  Annual net benefit: €{net_ben:,}")

print("\n" + "="*70)
print("\nTEMPLATE: Fill in your analysis")
print("""
Workflow chosen: _________________________________

Implementation cost: €_________________
Annual net benefit: €_________________

Calculations:
  Payback period: ______________ months
  ROI: ______________ %
  NPV (5-year @ 10%): €__________________

Recommendation: BUILD / SKIP / MAYBE
Reason: _________________________________________________________
""")

print("="*70)
print("\nANSWER KEY:")
for name, data in exercise_workflows.items():
    net_ben = data['ai_savings'] - data['ai_cost'] - data['support'] - data['risk']
    pb = payback_months(data['impl_cost'], net_ben)
    r = roi(data['impl_cost'], net_ben)
    n = npv(data['impl_cost'], [net_ben] * 5, 0.10)
    rec, reason = investment_recommendation(pb, r, n)
    print(f"\n{name}:")
    print(f"  Annual net benefit: €{net_ben:,.0f}")
    print(f"  Payback: {pb:.0f} months | ROI: {r:.1f}% | NPV: €{n:,.0f}")
    print(f"  → {rec}")

---

## Completion & Next Steps

### ✓ Lesson 04 Learning Outcomes
You can now:
- ✓ Calculate payback period and interpret it
- ✓ Calculate ROI and compare to hurdle rates
- ✓ Calculate NPV accounting for time value of money
- ✓ Run confidence adjustments for execution risk
- ✓ Perform sensitivity analysis on key drivers
- ✓ Make go/no-go investment decisions

### Lesson 04 Completion Checklist
To pass this lesson, you must demonstrate:

1. ✓ I can calculate payback, ROI, and NPV for any workflow
2. ✓ I understand why bundles beat individual workflows
3. ✓ I can adjust returns for realistic confidence factors
4. ✓ I ran sensitivity analysis identifying key cost/benefit drivers
5. ✓ I can recommend GO/NO-GO using decision framework

### Next: Lesson 05 — Private Equity Value Creation Playbook

Once you master investment evaluation, learn how PE managers think:

**"How do we create value across an entire company portfolio?"**

Lesson 05 teaches the PE playbook: 10 AI value levers, operating leverage, cost take-out, value creation plans, and 100-day rapid transformation plans.

🚀 Ready for Lesson 05?